In [384]:
%cd ../..

/Users/langfeldera


In [387]:
import awswrangler as wr
from wag_toolkit.locations import Locations
import pandas as pd
import json
from collections import Counter

In [11]:
dr_pubs_grants = wr.s3.read_excel('s3://datalabs-data/funding_impact_measures/dr_grants/dr_pub_grant_links.xlsx')

In [4]:
dr_schemes = wr.s3.read_excel('s3://datalabs-data/funding_impact_measures/dr_grants/award_list_legacy.xlsx')

In [4]:
dr_schemes = wr.s3.read_excel('s3://datalabs-data/funding_impact_measures/dr_grants/DRscheme_mapping.xlsx',sheet_name='by_ref')

In [7]:
lmics = pd.read_excel('notebooks/discovery_research/DR geographies_AAP list.xlsx', sheet_name='LMIC list')
lmics = lmics['LMIC_dimensions'].tolist()

In [14]:
dr_pubs_grants = dr_pubs_grants.merge(dr_schemes[['Grant Reference','legacy','scheme']], how='left', left_on='Reference', right_on='Grant Reference')

In [17]:
dr_pubs_grants.dropna(subset='legacy', inplace=True)

In [19]:
subgroups = dr_pubs_grants.groupby('scheme')['id'].apply(lambda pub_ids: list(set(list(pub_ids)))).to_dict()

In [21]:
pub_ids_l0 = list(set(dr_pubs_grants['id'].tolist()))

In [22]:
subgroups.keys()

dict_keys(['Career Development Award', 'Directed', 'Discovery Award', 'Early-Career Award', 'Off Strategy', 'Other', 'PhD'])

In [324]:
subgroup = 'PhD'
pub_ids_l0 = subgroups[subgroup]

In [325]:
len(pub_ids_l0)

1470

In [326]:
locations = Locations.from_publication_ids(pub_ids_l0, max_chunk_size=1000)

100%|██████████| 2/2 [00:09<00:00,  4.65s/it]


In [327]:
locations_raw = locations.data

In [328]:
locations.extract(location_level="institution")

100%|██████████| 2/2 [00:00<00:00,  5.85it/s]
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavi

2016
2017
2021
2015


/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.cr

2022
2019
2020
2023
2013


/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.cr

2018
2012
2011
2014
2010


/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.cr

In [329]:
publication_level_data = locations.publication_data

In [330]:
institution_lookup = locations.location_names

In [331]:
locations.data = locations_raw
locations._clean_grid_ids()

In [332]:
# How many authors with >1 grid id?
locations.data[locations.data['grid_id'].apply(lambda i: len(i))>1]

,dimensions_publication_id,year,grid_id


In [333]:
locations.extract_locations()

100%|██████████| 2/2 [00:00<00:00,  5.11it/s]


In [334]:
country_lookup = locations.location_names

In [335]:
mapping = {}
for grid_id, country in country_lookup.items():
    if country=='United Kingdom':
        mapping[institution_lookup[grid_id]] = 'uk'
    else:
        if country in lmics:
            mapping[institution_lookup[grid_id]] = 'lmic'
        else:
            mapping[institution_lookup[grid_id]] = 'hic'

In [336]:
mapping_reversed = {}
for grid_id, group in mapping.items():
    if mapping_reversed.get(group):
        mapping_reversed[group].append(grid_id)
    else:
        mapping_reversed[group] = [grid_id]

In [337]:
def group_adjacency_matrices(adjacency_matrices):
    grouped_matrices = {}
    for year in adjacency_matrices.keys():
        adj_matrix = adjacency_matrices[year]['All']
        adj_matrix['total'] = adj_matrix.loc['total',:]
        adj_matrix['group'] = adj_matrix.index.map(mapping)
        adj_matrix.loc['All', 'group'] = 'All'
        adj_matrix.loc['total', 'group'] = 'total'
        gm = adj_matrix.groupby('group').sum()
        for k in mapping_reversed.keys():
            if k not in gm.index:
                gm.loc[k, :] = 0
        grouped_matrices[year] = gm
    return grouped_matrices

In [338]:
grouped_adjacency_matrices = group_adjacency_matrices(locations.adjacency_matrices)

In [339]:
uk_institutes = []
for year, am in grouped_adjacency_matrices.items():
    am = am[[c for c in am.columns if c in mapping_reversed['uk']]]
    if year>=2013:
        am = am.transpose().sort_values('lmic', ascending=False)
        am.reset_index(inplace=True)
        am.rename(columns={'l2':'institute'}, inplace=True)
        am['year'] = year
        for group in ['lmic','hic','uk']:
            am[f'{group}_pct'] = am[group] / am['All'] *100
        uk_institutes.append(am)
uk_institutes = pd.concat(uk_institutes)

In [340]:
lmic_institutes = []
for year, am in grouped_adjacency_matrices.items():
    am = am[[c for c in am.columns if c in mapping_reversed['lmic']]]
    if year>=2013:
        am = am.transpose().sort_values('uk', ascending=False)
        am['year'] = year
        am.reset_index(inplace=True)
        am.rename(columns={'l2':'institute'}, inplace=True)
        for group in ['lmic','hic','uk']:
            am[f'{group}_pct'] = am[group] / am['All'] *100
        lmic_institutes.append(am)
lmic_institutes = pd.concat(lmic_institutes)

In [341]:
uk_all = uk_institutes.groupby('institute').sum().drop('year', axis=1).reset_index().sort_values('lmic', ascending=False)
lmic_all = lmic_institutes.groupby('institute').sum().drop('year', axis=1).reset_index().sort_values('uk', ascending=False)

In [342]:
top30_uk = list(reversed(uk_all.head(30)['institute'].tolist()))

In [343]:
top30_lmic = list(reversed(lmic_all.head(30)['institute'].tolist()))

In [344]:
#uk_institutes.to_excel('notebooks/discovery_research/uk_institutions.xlsx', sheet_name='uk_institutes', index=False)

In [345]:
#lmic_institutes.to_excel('notebooks/discovery_research/lmic_institutions.xlsx', sheet_name='lmic_institutes', index=False)

In [346]:
uk_data = {}
uk_data_pct = {}
for year in uk_institutes['year'].unique():
    df = uk_institutes[(uk_institutes['year']==year) & (uk_institutes['institute'].isin(top30_uk))]
    for i in set(top30_uk).difference(set(df['institute'].tolist())):
        df = pd.concat([df, pd.DataFrame({'institute':[i],
                                     'All':[0],
                                     'lmic':[0],
                                     'hic':[0],
                                     'total':[0],
                                     'uk':[0],
                                     'year':[year],
                                     'lmic_pct':[0],
                                     'hic_pct':[0],
                                     'uk_pct':[0]})])
    sort_order = {inst: order for order, inst in enumerate(top30_uk)}
    df['sort_order'] = df['institute'].map(sort_order)
    df = df.sort_values(by='sort_order')
    uk_data[str(year)] = [
        {
            'x': df['lmic'].astype(int).tolist(),
            'y': df['institute'].tolist(),
            'name': 'LMIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(21,113,242,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['hic'].astype(int).tolist(),
            'y': df['institute'].tolist(),
            'name': 'other HIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(106,160,53,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['uk'].astype(int).tolist(),
            'y': df['institute'].tolist(),
            'name': 'UK',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(255,195,0,0.7)',
                'width': 1
            },
            'type': 'bar'
        }
    ]
    uk_data_pct[str(year)] = [
        {
            'x': df['lmic_pct'].tolist(),
            'y': df['institute'].tolist(),
            'name': 'LMIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(21,113,242,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['hic_pct'].tolist(),
            'y': df['institute'].tolist(),
            'name': 'other HIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(106,160,53,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['uk_pct'].tolist(),
            'y': df['institute'].tolist(),
            'name': 'UK',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(255,195,0,0.7)',
                'width': 1
            },
            'type': 'bar'
        }
    ]

In [347]:
with(open(f'notebooks/discovery_research/barcharts/uk_institutions{"_"+subgroup}.json','w')) as f:
    json.dump(uk_data, f)

In [348]:
with(open(f'notebooks/discovery_research/barcharts/uk_institutions_pct{"_"+subgroup}.json','w')) as f:
    json.dump(uk_data_pct, f)

In [349]:
lmic_institutes.head()

group,institute,All,hic,lmic,total,uk,year,lmic_pct,hic_pct,uk_pct
0,Oxford University Clinical Research Unit,362.0,60.0,252.0,22.0,50.0,2016,69.613260,16.574586,13.812155
1,Aravind Eye Hospital,66.0,0.0,30.0,6.0,36.0,2016,45.454545,0.000000,54.545455
2,Papua New Guinea Institute of Medical Research,197.0,147.0,14.0,4.0,36.0,2016,7.106599,74.619289,18.274112
3,Addis Ababa University,80.0,3.0,47.0,7.0,30.0,2016,58.750000,3.750000,37.500000
4,Stellenbosch University,44.0,1.0,17.0,5.0,26.0,2016,38.636364,2.272727,59.090909


In [350]:
lmic_data = {}
lmic_data_pct = {}
for year in lmic_institutes['year'].unique():
    df = lmic_institutes[(lmic_institutes['year']==year) & (lmic_institutes['institute'].isin(top30_lmic))]
    for i in set(top30_lmic).difference(set(df['institute'].tolist())):
        df = pd.concat([df ,pd.DataFrame({'institute':[i],
                                     'All':[0],
                                     'lmic':[0],
                                     'hic':[0],
                                     'total':[0],
                                     'uk':[0],
                                     'year':[year],
                                     'lmic_pct':[0],
                                     'hic_pct':[0],
                                     'uk_pct':[0]})])
    sort_order = {inst: order for order, inst in enumerate(top30_lmic)}
    df['sort_order'] = df['institute'].map(sort_order)
    df = df.sort_values(by='sort_order')
    lmic_data[str(year)] = [
        {
            'x': df['uk'].astype(int).tolist(),
            'y': df['institute'].tolist(),
            'name': 'UK',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(255,195,0,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['lmic'].astype(int).tolist(),
            'y': df['institute'].tolist(),
            'name': 'LMIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(21,113,242,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['hic'].astype(int).tolist(),
            'y': df['institute'].tolist(),
            'name': 'other HIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(106,160,53,0.7)',
                'width': 1
            },
            'type': 'bar'
        }
    ]
    lmic_data_pct[str(year)] = [
        {
            'x': df['uk_pct'].tolist(),
            'y': df['institute'].tolist(),
            'name': 'UK',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(255,195,0,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['lmic_pct'].tolist(),
            'y': df['institute'].tolist(),
            'name': 'LMIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(21,113,242,0.7)',
                'width': 1
            },
            'type': 'bar'
        },
        {
            'x': df['hic_pct'].tolist(),
            'y': df['institute'].tolist(),
            'name': 'other HIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(106,160,53,0.7)',
                'width': 1
            },
            'type': 'bar'
        }
    ]

In [351]:
with(open(f'notebooks/discovery_research/barcharts/lmic_institutions{"_"+subgroup}.json','w')) as f:
    json.dump(lmic_data, f)

In [352]:
with(open(f'notebooks/discovery_research/barcharts/lmic_institutions_pct{"_"+subgroup}.json','w')) as f:
    json.dump(lmic_data_pct, f)

In [353]:
if subgroup != 'All':
    grants_info = dr_pubs_grants[dr_pubs_grants['scheme']==subgroup]
else:
    grants_info = dr_pubs_grants

In [354]:
grants_df = locations.publication_data.merge(grants_info[['id','Reference']], left_on='dimensions_publication_id', right_on='id')

In [355]:
mapping = {}
for country in country_lookup.values():
    if country == 'United Kingdom':
        mapping[country] = 'uk'
    elif country in lmics:
        mapping[country] = 'lmic'
    else:
        mapping[country] = 'hic'

In [356]:
grants_df['group'] = grants_df['location'].map(mapping)

In [357]:
lmic_wellcome_authorships = {}
for year in grants_df['year'].unique():
    if year>=2013:
        df = grants_df[grants_df['year']==year]
        lmic_wellcome_authorships[year]=df[df['group']=='lmic'].groupby('location')['Reference'].count().reset_index()

In [358]:
lmic_wellcome_all = grants_df[grants_df['group']=='lmic'].groupby('location')['Reference'].count().sort_values(ascending=False)

In [359]:
lmic_wellcome_top30 = list(reversed(lmic_wellcome_all.reset_index().head(30)['location'].tolist()))

In [360]:
wellcome_authorships_data = {}
for year, counts in lmic_wellcome_authorships.items():
    df = counts[counts['location'].isin(lmic_wellcome_top30)]
    for i in set(lmic_wellcome_top30).difference(set(df['location'].tolist())):
        df = pd.concat([df ,pd.DataFrame({'location':[i],
                                     'Reference':[0]})])
    sort_order = {inst: order for order, inst in enumerate(lmic_wellcome_top30)}
    df['sort_order'] = df['location'].map(sort_order)
    df = df.sort_values(by='sort_order')
    wellcome_authorships_data[str(year)] = [
            {
                'x': df['Reference'].astype(int).tolist(),
                'y': df['location'].tolist(),
                'name': 'Number of authorships',
                'orientation': 'h',
                'marker': {
                    'color': 'rgba(21,113,242,0.7)',
                    'width': 1
                },
                'type': 'bar'
            }
        ]

In [361]:
with(open(f'notebooks/discovery_research/barcharts/lmic_wellcome_authorships{"_"+subgroup}.json','w')) as f:
    json.dump(wellcome_authorships_data, f)

In [362]:
lmic_unique_grant_authorships = {}
for year in grants_df['year'].unique():
    if year>=2013:
        df = grants_df[grants_df['year']==year]
        lmic_unique_grant_authorships[year] = df[df['group']=='lmic'].groupby('location')['Reference'].nunique().reset_index()

In [363]:
lmic_grant_all = grants_df[grants_df['group']=='lmic'].groupby('location')['Reference'].nunique().sort_values(ascending=False)

In [364]:
lmic_grant_top30 = list(reversed(lmic_grant_all.reset_index().head(30)['location'].tolist()))

In [365]:
grant_authorships_data = {}
for year, counts in lmic_unique_grant_authorships.items():
    df = counts[counts['location'].isin(lmic_grant_top30)]
    for i in set(lmic_grant_top30).difference(set(df['location'].tolist())):
        df = pd.concat([df ,pd.DataFrame({'location':[i],
                                     'Reference':[0]})])
    sort_order = {inst: order for order, inst in enumerate(lmic_grant_top30)}
    df['sort_order'] = df['location'].map(sort_order)
    df = df.sort_values(by='sort_order')
    grant_authorships_data[str(year)] = [
            {
                'x': df['Reference'].astype(int).tolist(),
                'y': df['location'].tolist(),
                'name': 'Number of unique grants',
                'orientation': 'h',
                'marker': {
                    'color': 'rgba(21,113,242,0.7)',
                    'width': 1
                },
                'type': 'bar'
            }
        ]

In [366]:
with(open(f'notebooks/discovery_research/barcharts/lmic_grant_authorships{"_"+subgroup}.json','w')) as f:
    json.dump(grant_authorships_data, f)

In [367]:
grant_titles = pd.read_excel('notebooks/discovery_research/DR_overview_dash_export.xlsx', sheet_name='Overview')

In [368]:
grant_titles = grant_titles.set_index('Reference')['Title'].to_dict()

In [369]:
grants_df['grant_title'] = grants_df['Reference'].map(grant_titles)
#grants_df.to_excel('notebooks/discovery_research/grants.xlsx', sheet_name='grants', index=False)

In [370]:
#grants_df[grants_df['group']=='china']['grant_title'].value_counts().to_csv('notebooks/discovery_research/china_grants.csv')

In [371]:
#grants_df[grants_df['group']=='china']

In [372]:
grant_matrices = {}
for year in grants_df['year'].unique():
    if year>=2013:
        df = grants_df[grants_df['year']==year]
        am = pd.crosstab(df['Reference'], df['group']).reset_index()
        am['title'] = am['Reference'].map(grant_titles)
        grant_matrices[year] = am

In [373]:
grant_lmic_data = {}
for year, df in grant_matrices.items():
    df = df.sort_values(by='lmic', ascending=False).head(30)
    df = df.reset_index()[::-1]
    grant_lmic_data[str(year)] = [
            {
            'x': df['lmic'].astype(int).tolist(),
            'y': df['Reference'].tolist(),
            'name': 'LMIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(21,113,242,0.7)',
                'width': 1
            },
            'text': [f'Title: {t}' for t in df['title'].tolist()],
            'hovertemplate': "%{x}, %{text}",
            'textposition': "none",
            'type': 'bar'
        },
        {
            'x': df['hic'].astype(int).tolist(),
            'y': df['Reference'].tolist(),
            'name': 'other HIC',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(106,160,53,0.7)',
                'width': 1
            },
            'text': [f'Title: {t}' for t in df['title'].tolist()],
            'hovertemplate': "%{x}, %{text}",
            'textposition': "none",
            'type': 'bar'
        },
        {
            'x': df['uk'].astype(int).tolist(),
            'y': df['Reference'].tolist(),
            'name': 'UK',
            'orientation': 'h',
            'marker': {
                'color': 'rgba(255,195,0,0.7)',
                'width': 1
            },
            'text': [f'Title: {t}' for t in df['title'].tolist()],
            'hovertemplate': "%{x}, %{text}",
            'textposition': "none",
            'type': 'bar'
        }
        ]

In [321]:
with(open(f'notebooks/discovery_research/barcharts/grant_lmic_data{"_"+subgroup}.json','w')) as f:
    json.dump(grant_lmic_data, f)

In [16]:
locations.data = locations_raw
locations.extract_locations(location_level='institution')
institution_lookup = locations.location_names

100%|██████████| 9/9 [00:02<00:00,  4.19it/s]


In [18]:
locations.data = locations_raw
locations.extract_edges()

100%|██████████| 25/25 [00:22<00:00,  1.13it/s]


In [29]:
def convert_edges(coauthorship_edges, country_names, institution_names, lmics):
    converted_edges = {}
    for year, coauthorship_edges in coauthorship_edges.items():
        converted_edges[year] = Counter()
        for edge, weight in coauthorship_edges.items():
            grid_id_0, grid_id_1 = edge
            country_0 = country_names.get(grid_id_0)
            country_1 = country_names.get(grid_id_1)
            if (country_0=='United Kingdom' and (country_1 in lmics)):
                converted_edges[year].update({(institution_names.get(grid_id_0), country_1): weight})
            elif (country_1=='United Kingdom' and (country_0 in lmics)):
                converted_edges[year].update({(institution_names.get(grid_id_1), country_0): weight})
    return converted_edges

In [30]:
uk_lmic_edges = convert_edges(locations.coauthorship_edges, country_lookup, institution_lookup, lmics)

In [32]:
locations.publication_data

,dimensions_publication_id,year,grid_id,location
0,pub.1021311596,2012,grid.451056.3,National Institute for Health Research
1,pub.1021311596,2012,grid.451056.3,National Institute for Health Research
2,pub.1021311596,2012,grid.5330.5,University of Erlangen-Nuremberg
3,pub.1021311596,2012,grid.414699.7,Rotterdam Eye Hospital
4,pub.1021311596,2012,grid.1008.9,University of Melbourne
...,...,...,...,...
607506,pub.1135331548,2021,grid.451388.3,The Francis Crick Institute
607507,pub.1135331548,2021,grid.451388.3,The Francis Crick Institute
607508,pub.1135331548,2021,grid.451388.3,The Francis Crick Institute
607509,pub.1135331548,2021,grid.7700.0,Heidelberg University


In [374]:
pub_ids_l0 = list(set(dr_pubs_grants['id'].tolist()))

In [375]:
locations = Locations.from_publication_ids(pub_ids_l0, max_chunk_size=1000)

100%|██████████| 51/51 [00:43<00:00,  1.18it/s]


In [376]:
locations.extract()

100%|██████████| 9/9 [00:01<00:00,  4.90it/s]


2022
2018
2012
2019
2021
2016


/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.cr

2020
2015
2014
2017
2023
2010
2013
2008


/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.cr

2009
2011
2007
2005
2006
2003
2004
2001
2002
1989


/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.cr

In [377]:
locations.load_visjs_nodes_and_edges(node_scaling=0.0015, edge_scaling=0.001, directed=False, threshold=0, node_count="total", font = {"size": 20, "face": "Helvetica Neue"})

In [378]:
locations.to_visjs(vis_name="locations", directed=False)

In [382]:
locations._to_json("nodes_all.json", locations.vis_nodes)
locations._to_json("edges_all.json", locations.vis_edges)

Add subgroups

In [383]:
for subgroup, pub_ids in subgroups.items():
    print(subgroup)
    locations = Locations.from_publication_ids(pub_ids, max_chunk_size=1000)
    locations.extract()
    locations.load_visjs_nodes_and_edges(node_scaling=0.0015, edge_scaling=0.001, directed=False, threshold=0, node_count="total", font = {"size": 20, "face": "Helvetica Neue"})
    locations._to_json(f"nodes_{subgroup}.json", locations.vis_nodes)
    locations._to_json(f"edges_{subgroup}.json", locations.vis_edges)

Career Development Award


100%|██████████| 5/5 [00:00<00:00,  7.74it/s]


2018
2022
2016
2019
2021
2020
2017
2015
2012


/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.cr

2014
2023
2013
2010
2011
2009
2008
1989


...

2002
2023
2012
2003
2001
Other


100%|██████████| 1/1 [00:00<00:00,  6.99it/s]
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavi

2015
2018
2019
2017
2021
2022
2016
2020
PhD


100%|██████████| 2/2 [00:00<00:00,  7.22it/s]
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/langfeldera/github_repos/wellcome_academic_graph/toolkit/locations.py:176: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavi

2016
2017
2021
2015
2022
2019
2020
2023
2013
2018
2012
2011
2014
2010


Add positions based on latitude and longitude

In [123]:
lat_lon = pd.read_csv('notebooks/discovery_research/countries.csv')

In [124]:
lat_lon.rename(columns={'Latitude (average)':'y', 'Longitude (average)':'x'}, inplace=True)

In [125]:
lat_lon.dropna(subset=['x','y'], inplace=True)

In [126]:
lat_lon['y'] = lat_lon['y']*-1

In [127]:
lat_lon['x'] = lat_lon['x']*25
lat_lon['y'] = lat_lon['y']*25

In [128]:
positions = lat_lon.set_index('Country')[['x','y']].to_dict('index')

In [129]:
with(open('notebooks/discovery_research/positions.json','w')) as f:
    json.dump(positions, f)

In [87]:
for c in set(locations.location_names.values()):
    if c not in positions.keys():
        print(c)

In [19]:
#wr.s3.to_parquet(locations.publication_data, 's3://datalabs-data/funding_impact_measures/dr_publications/dr_pubs_countries.parquet')

{'paths': ['s3://datalabs-data/funding_impact_measures/dr_publications/dr_pubs_countries.parquet'],
 'partitions_values': {}}

In [35]:
with(open('notebooks/location_vis/edges.json','r')) as f:
    edges = json.load(f)

In [388]:
def edges_to_dict(edges):
    edge_dict = {}
    for e in edges:
        year = e['year']
        _ = e.pop('year')
        if edge_dict.get(year):
            edge_dict[year].append(e)
        else:
            edge_dict[year] = [e]
    return edge_dict

In [3]:
edge_fnames = ['edges_all.json', 'edges_cd.json', 'edges_da.json', 'edges_dir.json', 'edges_ec.json', 'edges_other.json']

In [391]:
for sg in subgroups.keys():
    with(open(f'notebooks/location_vis//new/edges_{sg}.json','r')) as f:
        edges = json.load(f)
        for e in edges:
            weight = e['title'].split(': ')[-1].replace(',','')
            e['weight'] = int(weight)
    edge_dict = edges_to_dict(edges)
    with(open(f'notebooks/location_vis/new/dict_edges_{sg}.json','w')) as f:
        json.dump(edge_dict, f)

In [6]:
node_fnames = ['nodes_all.json', 'nodes_cd.json', 'nodes_da.json', 'nodes_dir.json', 'nodes_ec.json', 'nodes_other.json']

In [7]:
with(open(f'notebooks/location_vis/{node_fnames[0]}','r')) as f:
    nodes = json.load(f)

In [392]:
min_size = 5

In [14]:
for data in nodes.values():
    for d in data:
        d['size'] += (min_size-1)

In [393]:
# Adjust node size
for sg in subgroups.keys():
    with(open(f'notebooks/location_vis/new/nodes_{sg}.json','r')) as f:
        nodes = json.load(f)
    for data in nodes.values():
        for d in data:
            d['size'] += (min_size-1)
    with(open(f'notebooks/location_vis/new/nodes_{sg}.json','w')) as f:
        json.dump(nodes, f)

In [138]:
with(open('notebooks/discovery_research/edges.json','w')) as f:
    json.dump(edges, f)